# Causal Inference Fundamentals and Randomized Control Trials (RCT)

*“Between a good and a bad economist this constitutes the whole difference: the one takes account only of the visible effect; the other takes account of both the effects which are seen and those which it is necessary to foresee.”*
<br>
-Frederic Bastiat
<br><br>
This lesson has three goals:
- Introduce the [Potential Outcomes Framework](#Potential-Outcomes-Framework) and [Selection Bias](#Selection-Bias)
- Explain why [Randomization Solves Selection Bias](#Randomization)
- Discuss tips for [Presenting/Checking RCT Results](#RCT-Presentation)
    
As data scientists, a tremendous amount of our work regards explaining why outcomes occur. The most common technique we use are Randomized Control Trials, otherwise known as A/B Tests. Setting up and interpretting these experiments is straightforward, but relatively few practitioners have a strong understanding of the fundamentals underlying this technique. 

This lesson introduces the theoretical framework underlying virtually *all* causal inference techniques, using the RCT method to illustrate its points. Not only will it better prepare you to think about and discuss A/B Tests with your stakeholders, but it will provide a foundation for thinking about the ever-changing bevy of causal inference techniques that exist beyond RCT.
<br>

<a id='Potential-Outcomes-Framework'></a>
## The Potential Outcomes Framework

We all have an intuitive understanding of what it means for two things to be causally related; a change in one *causes* a change in the other. But it's less obvious how to define and measure causality mathematically. For this, scientists rely on the **Potential Outcomes Framework**.

To understand the potential outcomes framework, it's helpful to use a running example. Let's consider a scenario in which we are trying to improve customer retention by premptively calling customers when their usage is unexpectedly high, so that they are not "shocked" by their large bill. To this end, we decide to determine the efficacy of our program by measuring customer tenure across two groups of people: those who received preemptive calls, and those who did not. After gathering data, we do a bit of exploration and find the following:

| Group         | Average Tenure (months) |
| :-----------: | :---------------------: |
| Received Call |           19            |
| No Call       |           25            |
<br>

This is strange! Calling our customers should improve our relationship with them, resulting in longer tenure. Why, then, would individuals who received calls have shorter average tenure than those who do not? The answer is that the relationship in the table above is not causal; there is more that determines a customer's tenure than whether or not they receive a preemptive call. Specifically, customers who receive calls are probably more likely to have shorter tenure to begin with, since they experience "bill shock." 

With this in mind, we can be a bit more explicit about what our comparison above is quantifying. Because customers who receive calls also have unusually high energy bills, this table compares average tenure between customers who receive a call *and experience bill shock*, against customers who do not receive a call *and do not experience bill shock*. Thus, the difference in average tenure between the two groups reflects the combined effects of our call phone program and bill shock. Obviously we would like to disentangle these two effects so that we can focus on just the effect of our call program. However, using only our *observed* data in the table above, this might not be possible. 

This all leads us back to the question we began with: how can we measure the efficacy of our preemptive call program?

<br>
This line of inquiry leads us to the potential outcomes framework. To begin, let's clearly define the comparisons we would like to make in this example in order to determine the causal effect of our program, but can't. Specificially, there are two comparisons that our table suggests could help inform us about the causal effect of our program: 

1. We could compare the tenure of customers who receive phone calls to *what their tenure would have been, had they not received a phone call*.
2. We could compare the tenure of customers who do not receive phone calls to *what their tenure would have been, had they received a phone call*.

Importantly, these are counterfactual comparisons. By this I mean that they evaluate the difference between an observed outcome and a hypothetical outcome that would have occured had we taken a different course of action. By making the leap from observed to counterfactual comparisons, we are able to disentangle the effects of our call program and bill shock. Specifically, our first counterfactual comparison quantifies the difference in tenure between customers who receive a call and customers who do not receive a phone call, for customers experiencing bill shock. Our second counterfactual comparison does the same, but for customers that do not experience bill shock.

Now that we have a conceptual framework for the comparisons we would like to make, let's define them mathematically. This will serve as our introduction to the potential outcomes framework. Let's start by using $Y$ to describe each customer's outcome (their tenure), and $D$ as a dummy variable to denote whether they received a preemptive call. Each customer's *observed* potential outcome is then:

$$ 
\text{Obeserved Potential Outcome} =   
\left\{
   \begin{array}{ll}
       Y_{1i} & \text{ if } D_{i} = 1 \\
       Y_{0i} & \text{ if } D_{i} = 0
   \end{array} 
\right.
$$

While we only observe $Y_{1i}$ and $Y_{0i}$ when $D_{i} = 1$ and $D_{i} = 0$, respectively, in principle they do not depend on treatment status.
> **Note:** The word "treatment" refers to the variable whose causal effect we are trying to quantify; in this case "treatment" refers to whether an individual is selected to receive a preemptive call. Using our established notation, $D_{i} = 1$ for treated individuals, and $D_{i} = 0$ for the untreated. Causal inference practitioners use the word "treatment" ubiquitously when describing their studies, so it's a good idea to get comfortable using it yourself. 

Thus, $Y_{1i}$ is the tenure of a customer who received a call, or what their tenure would have been if they had received one. Meanwhile, $Y_{0i}$ represents the tenure of a customer who did not receive a call, or what their tenure would have been had they had not received one. Following this line of reasoning, each customer also has a *counterfactual* potential outcome: 

$$ 
\text{Counterfactual Potential Outcome} =   
\left\{
   \begin{array}{ll}
       Y_{0i} & \text{ if } D_{i} = 1 \\
       Y_{1i} & \text{ if } D_{i} = 0
   \end{array} 
\right.
$$

Ideally, we would like to be able to measure both the observed and counterfactual outcomes for the same individual. Looking back at our two counterfactual comparisons, we can see that the quantity we are really interested in is:

\begin{equation*}
    \text{Causal Effect}_{i} = Y_{1i} - Y_{0i} 
\end{equation*}

If we could calculate this difference, we would be explicitly making one of our two counterfactual comparisons for each customer. In turn, this would allow us to determine the causal impact of our preemptive call program. The problem, of course, is that we can't observe both of these outcomes simultaneously. To do this, we would need to overcome the metaphysical challenge of observing the same individual in two parallel universes: one in which they never receive a preemptive call, and another in which they do receive a preemptive call. Clearly, this isn't something that machine learning algorithms can yet accomplish.

<a id='Selection-Bias'></a>
## Selection Bias

<br>
So, if we can't measure causal effects at the individual level, what can we do? Typically, we set our sights on measuring an average causal effect by examining groups of individuals. Luckily, we've already started thinking along these lines by using our table to compare average tenure by group. Let's revisit these values using the notation we've built up so far. This will enable us to more precisely understand why the comparison is not causal, and what we can do about it. In particular, note that our naive table comparison shows us the following:
<br><br>
\begin{equation*}
    \underbrace{E[Y_{1i}\text{ } | \text{ } D_{i} = 1] - E[Y_{0i} \text{ } | \text{ } D_{i} = 0]}_\text{Observed difference in average tenure}
\end{equation*}
<br>

In words, this equation says that we are evaluating the average difference in tenure between people who received calls and people who did not, *conditional on these groups' observed call status*. Remember, a causal interpretation requires us to incorporate the *counterfactual potential outcome*. The good news is that if we add a "clever zero" to the equation, we can see how we might do just that. In particular, let's add the following "zero" to our naive comparison: $E[Y_{0i} \text{ } | \text{ } D_{i} = 1] - E[Y_{0i} \text{ } | \text{ } D_{i} = 1] = 0$. Doing so, we can link the equation for our table's naive comparison to one with a causal interpretation as follows:
<br><br>
\begin{equation}
    \underbrace{E[Y_{1i}\text{ } | \text{ } D_{i} = 1] - E[Y_{0i} \text{ } | \text{ } D_{i} = 0]}_\text{Observed difference in average tenure} = \underbrace{E[Y_{1i}\text{ } | \text{ } D_{i} = 1] - E[Y_{0i} \text{ } | \text{ } D_{i} = 1]}_\text{Average treatment effect on the treated} + \underbrace{E[Y_{0i}\text{ } | \text{ } D_{i} = 1] - E[Y_{0i} \text{ } | \text{ } D_{i} = 0]}_\text{Selection bias}
\end{equation}
>Note that the average treatment effect on the treated simplifies to: $ \underbrace{E[Y_{1i} - Y_{0i} \text{ } | \text{ } D_{i} = 1]}_\text{Average treatment effect on the treated}$

>You should notice that this looks very similar to the causal effect equation in the previous cell, with the difference being that it is an average effect conditional on $D_{i} = 1$.

<br>
The term $E[Y_{1i}\text{ } | \text{ } D_{i} = 1] - E[Y_{0i} \text{ } | \text{ } D_{i} = 1]$ is called the "average treatment effect on the treated", and is the average causal effect of preemptive calls, for those who are selected to receive calls. More specifically, it reflects the average difference in observed tenure for customers who received calls $E[Y_{1i}\text{ } | \text{ } D_{i} = 1]$, and what their tenure would have been had they not received a call $E[Y_{0i}\text{ } | \text{ } D_{i} = 1]$. Bringing things full circle, this term is just a mathematical representation of our first counterfactual comparison.
<br><br>
Importantly, we can now see that our naive comparison adds a component to our causal comparison. In particular, the term $E[Y_{0i}\text{ } | \text{ } D_{i} = 1] - E[Y_{0i} \text{ } | \text{ } D_{i} = 0]$ is called selection bias, and represents the average difference in _uncalled_ tenure between those who were selected to receive preemptive calls and those who were not. Because customers who are selected to receive preemptive calls have on average a shorter tenure than those who are not, selection bias is negative in this example. Selection bias can be so large in magnitude that it overpowers the causal effect; this is why our naive comparison indicates that customers who receive preemptive phone calls have shorter tenure than those who do not, even though preemptive phone calls probably extend customer tenure.
<br><br>
In virtually any causal inference project, the first thing you should think about it how to eliminate selection bias. If you can't come up with a comparison that gets rid of selection bias, you can't make any claim of causality. In that spirit, the succeeding sections of this notebook will describe how RCTs accomplish this goal.

<a id='Randomization'></a>
## Randomization Solves Selection Bias

At this point you should be able to see why selection bias is a problem, but it might not be obvious how to avoid it. As a reminder, our goal in this exercise is to calculate the average treatment effect on the treated (ATT). Rearranging our formula from the previous section, we have:

\begin{equation}
\underbrace{E[Y_{1i}\text{ } | \text{ } D_{i} = 1] - E[Y_{0i} \text{ } | \text{ } D_{i} = 1]}_\text{Average treatment effect on the treated} = \underbrace{E[Y_{1i}\text{ } | \text{ } D_{i} = 1] - E[Y_{0i} \text{ } | \text{ } D_{i} = 0]}_\text{Observed difference in average tenure} - \underbrace{E[Y_{0i}\text{ } | \text{ } D_{i} = 1] - E[Y_{0i} \text{ } | \text{ } D_{i} = 0]}_\text{Selection bias}
\end{equation}

In order to calculate the ATT, we could try to ensure that $E[Y_{0i}\text{ } | \text{ } D_{i} = 1] = E[Y_{0i} \text{ } | \text{ } D_{i} = 0]$, so that the selection bias term is zero. This is nice because it means our observed difference has a causal interpretation, but it requires randomization. To understand why randomization allows us to eliminate selection bias, note that randomization ensures that treatment status ($D_{i}$) is independent of a customer's potential outcomes:
\begin{equation*}
    \{Y_{0i}, Y_{1i}\} \perp\!\!\!\!\!\perp  D_{i}
\end{equation*}
Thus, if we randomly decide which customers receive phone calls: 
\begin{equation*}
E[Y_{0i} \text{ } | \text{ } D_{i} = 0] = E[Y_{0i} \text{ } | \text{ } D_{i} = 1] = E[Y_{0i}]
\end{equation*}
In this case, our Selection Bias term disappears:
\begin{align*}
    \text{Selection bias} &= E[Y_{0i}\text{ } | \text{ } D_{i} = 1] - E[Y_{0i} \text{ } | \text{ } D_{i} = 0] \\
    &= E[Y_{0i}] - E[Y_{0i}] \\
    &= 0
\end{align*}

By extension, our observed difference is equal to the ATT. Putting the pieces together, we can explicitly show the effect of randomization as follows:

\begin{align*}
    \underbrace{E[Y_{1i}\text{ } | \text{ } D_{i} = 1] - E[Y_{0i} \text{ } | \text{ } D_{i} = 1]}_\text{Average treatment effect on the treated} &= \underbrace{E[Y_{1i}\text{ } | \text{ } D_{i} = 1] -E[Y_{0i} \text{ } | \text{ } D_{i} = 0]}_\text{Observed difference in average tenure} - \underbrace{E[Y_{0i}\text{ } | \text{ } D_{i} = 1] - E[Y_{0i} \text{ } | \text{ } D_{i} = 0]}_\text{Selection bias} \\ 
    \underbrace{E[Y_{1i}\text{ } | \text{ } D_{i} = 1] - E[Y_{0i} \text{ } | \text{ } D_{i} = 1]}_\text{Average treatment effect on the treated} &= \underbrace{E[Y_{1i}\text{ } | \text{ } D_{i} = 1] -E[Y_{0i} \text{ } | \text{ } D_{i} = 0]}_\text{Observed difference in average tenure} - \underbrace{0}_\text{Selection bias} \\
    \underbrace{E[Y_{1i} - Y_{0i} \text{ } | \text{ } D_{i} = 1]}_\text{Average treatment effect on the treated} &= \underbrace{E[Y_{1i}\text{ } | \text{ } D_{i} = 1] - E[Y_{0i} \text{ } | \text{ } D_{i} = 0]}_\text{Observed difference in average tenure}
\end{align*}
<br><br>
That's it! Eliminating selection bias really is that simple. Although we can't know the effect a phone call has for a single individual, we can figure out the average effect for a larger portion of the population. Typically this is the subset of the population that is selected for treatment; hence why we name this quantity the **average treatment effect** ***on the treated***. However, in the case where we run a randomized experiment, we can actually go one step further. Because randomization allows us to drop the conditional component of our expectation, the causal effect we've estimated is true for our entire population. Stated mathematically, this means that randomly assigning treatment implies the following:
\begin{equation*}
    E[Y_{1i} \text{ } | \text{ } D_{i} = 1] = E[Y_{1i} \text{ } | \text{ } D_{i} = 0] = E[Y_{1i}]
\end{equation*}
<br>
Of course, we can also drop the conditional component of our expectation for those who do not receive treatment (we saw this a few lines ago when discussing why the selection bias term disappears). In combination, our equation for the ATT becomes:
\begin{equation*}
    \underbrace{E[Y_{1i} - Y_{0i}]}_\text{Average treatment effect} = \underbrace{E[Y_{1i}\text{ } | \text{ } D_{i} = 1] - E[Y_{0i} \text{ } | \text{ } D_{i} = 0]}_\text{Observed difference in average tenure}
\end{equation*}

<br><br>
Thus, our random experiment estimates the **average treatment effect (ATE)**, which might be a slightly different quantity than we expect. In our current context, this is the effect of a preemptive phone call for a typical customer, *not* the effect of a phone call for a customer with unexpectedly high usage. If you think about it, this should make sense: by randomly deciding who received a phone call in this example, we implicitly decided against exclusively calling high usage customers. The new interpretation of our causal estimate mirrors this change. 

<br>
However, given the business context of this example, the ATE might not be exactly what we want to estimate: our preemptive call program is meant to be focused on customers we think are going to experience bill shock, not our typical customer. So, how big of a concern is this? The answer is that it depends. We might believe that preemptive calls are actually more effective for customers with unexpectedly high bills. If this is the case, the ATE is probably a bit smaller than the ATT. In turn, presenting stakeholders with the ATE would be a conservative estimate of the impact of our call program. In the spirit of under-promising and overperforming, this could play to your advantage. On the other hand, we might not have any priors about the ATT. Risk averse stakeholders might not be comfortable utilizing the ATE to greenlight decisions in this context.

## Some Final Notes on RCTs

Randomized Control Trials are a great way to determine causal effects, but we've seen that we still need to be a bit cautious when utilizing them. There are a few main concerns that arise: 
- RCTs are often expensive and time consuming to run. Usually, you should consider whether or not observational data could be used in place of an experiment.
- You want to be sure that you're interested in the ATE, rather than the ATT, prior to running them. If you are interested in the ATT, make sure you randomize treatment _within the group of interest_. Interpret your results carefully. 
- RCTs are susceptible to concerns about "external validity." By this, I mean that the ATE estimated by one RCT might be very different than the ATE estimated by a similar RCT, carried out at a different time or in a different place. You should be careful not to overstate the results of your particular study: the causal effect you've found might not be true everywhere.

With these caveats in mind, let's talk about ways you can present (or at least check) your results to ensure you have a compelling case for your RCT's validity.

<a id='RCT-Presentation'></a>
## Presenting/Checking RCT Results

When you think about your RCT work, there are a few things you should be sure to do. First, you should actually provide evidence that you assigned treatment randomly. This might seem trivial, but it's utterly fundamental to identifying a causal effect. 

Remember, the point of randomization is to ensure that selection bias is eliminated. This is achieved by ensuring that our counterfactual potential outcome, though unobserved, is independent of treatment status. Since we don't observe the counterfactual potential outcome, we can't explicitly show that it's identitical to the control group's outcome. However, we can do the next best thing and show that treatment status is uncorrelated with anything else we can think of. This can be achieved by making a table of summary statistics for any/all variables that describe our treatment and control groups, and running tests of statistical significance on their difference in means. If your variables' means are quite close to one another, it's a good sign that treatment was indeed randomly assigned. Here's an example of what that might look like:

<center><img src="./images/example_balance_table.png" width=800 height=800 /></center>
</br>
Of course, your business stakeholders may not need to see a huge table of summary statistics. You'll have to decide for yourself whether you include this information in your presentation(s). Still, at a minimum you should  do this yourself in order to check your work.

Second, you should show the observed difference in means between your test and control groups. Keep in mind that you will also want to include a standard error on your estimate, so that you know whether it is statiscally significant. The easiest way to do this is by running the following regression:
\begin{equation*}
    y_{i} = \hat{\beta_{0}} + \hat{\beta_{1}}D_{i} + \epsilon_{i}
\end{equation*}
Where $\hat{\beta_{1}}$ is your estimated effect. Remember, because we have randomized:
\begin{equation*}
    \hat{\beta_{1}} = \underbrace{E[Y_{1i} - Y_{0i}]}_\text{Average treatment effect}
\end{equation*}

Be sure to interpret your results appropriately; this is the average treatment effect for your entire population, *not* the average treatment effect on the treated.

Third, and finally, you should check to be sure that adding additional controls to this regression does not change your ATE. If they do, it's likely that you did not successfully randomize which individuals received treatment. 

Given our discussion of the FWL theorem in the previous lesson, we know that adding variables to our regression that are uncorrelated with $D_{i}$ should not change $\hat{\beta_{1}}$. If we randomized correctly, any variable we think of should be uncorrelated with $D_{i}$. In fact, including these additional variables should have the benefit of reducing our standard error on $\hat{\beta_{1}}$, so long as they have some explanatory power for $y_{i}$.

One of the most common ways to accomplish this goal is to include a pre-intervention measure of the outcome variable. To understand why this is the case, let's return to our running example. There could be dozens of variables in our data that plausibly affect customers' tenure, and so we *could* add these myriad controls to our regression in order to test whether they impact our estimated average treatment effect. As an alternative, it might be simpler to simply include customer's pre-intervention tenure as a somewhat aggregate statistic for all these other factors. In this case, your regression will look like the following:
\begin{equation*}
    y_{it} = \hat{\beta_{0}} + \hat{\beta_{1}}D_{i} + \hat{\beta_{2}}y_{it-1} + \epsilon_{i}
\end{equation*}
which simplifies to the following if we normalize $\hat{\beta_{2}} = 1$:
\begin{equation*}
    y_{it} - y_{it-1} = \hat{\beta_{0}} + \hat{\beta_{1}}D_{i} + \epsilon_{i}
\end{equation*}